In [ ]:
!pwd


In [ ]:
!nvidia-smi


# Create Environment


In [ ]:
!test -d /content/edge-lipsync-model || git clone https://github.com/monkira99/edge-lipsync-model.git /content/edge-lipsync-model


In [ ]:
%cd /content/edge-lipsync-model


In [ ]:
!git pull


In [ ]:
!uv sync


In [ ]:
import os

try:
    from google.colab import userdata
    for key in ("HF_TOKEN", "WANDB_API_KEY"):
        value = userdata.get(key)
        if value:
            os.environ[key] = value
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or environment."
assert os.environ.get("WANDB_API_KEY"), "Set WANDB_API_KEY in Colab Secrets or environment."


In [ ]:
!hf auth login --token "$HF_TOKEN" --quiet
!uv run wandb login "$WANDB_API_KEY" --verify


In [ ]:
!uv run python tools/hf_model_assets.py pull \
  --repo-id tiennguyenbnbk/edge-lipsync-model-assets \
  --local-dir models


In [ ]:
!ls -lah /content/edge-lipsync-model/models/


# Training Configuration


In [ ]:
from pathlib import Path

HF_DATASET_REPO = "tiennguyenbnbk/hdtf-xdub-duix-dataset-SaxbyChambliss"
HF_MODEL_REPO = "tiennguyenbnbk/lipsync-SaxbyChambliss-v1"
RUN_DIR = Path("/content/data/runs/hdtf_speaker_baseline")
HF_CACHE_DIR = Path("/content/data/hf_cache")
WANDB_DIR = Path("/content/data/wandb")
LOCAL_DATASET_ROOT = Path("/content/data/hdtf_xdub_duix_dataset")

RUN_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WANDB_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /content/edge-lipsync-model/configs/train.colab.yaml
run_dir: /content/data/runs/hdtf_speaker_baseline

hf_dataset_repo: "tiennguyenbnbk/hdtf-xdub-duix-dataset-SaxbyChambliss"
hf_cache_dir: "/content/data/hf_cache"

init_bin: /content/edge-lipsync-model/models/emma/dh_model.bin
init_ckpt: ""
hf_init_model_repo: ""
hf_init_model_filename: best.pt

hf_model_repo: "tiennguyenbnbk/lipsync-SaxbyChambliss-v1"
hf_model_private: true

device: cuda
precision: mixed
batch_size: 64
num_workers: 4
learning_rate: 0.00001
weight_decay: 0.0001

max_steps: 10000
warmup_steps: 100
stabilization_steps: 10
stabilization_lr_scale: 0.1
validation_interval: 20
checkpoint_interval: 20
log_interval: 10

wandb_mode: online
wandb_project: edge-lipsync-model
wandb_entity: ""
wandb_run_name: hdtf-SaxbyChambliss-baseline-v1
wandb_group: hdtf-SaxbyChambliss
wandb_tags:
  - baseline
  - hdtf
  - single-speaker
  - duix-unet
wandb_notes: "Initial fine-tuning from Emma dh_model.bin"
wandb_dir: /content/data/wandb


In [ ]:
!sed -n "1,220p" /content/edge-lipsync-model/configs/train.colab.yaml


In [ ]:
!uv run python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')"


# Verify Dataset Load


In [ ]:
!uv run python tools/hf_dataset.py pull \
  --repo-id {HF_DATASET_REPO} \
  --cache-dir {HF_CACHE_DIR}


# Optional Before-Train Inference


In [ ]:
assert (LOCAL_DATASET_ROOT / "manifest.jsonl").exists(), (
    "LOCAL_DATASET_ROOT must point to a file-tree dataset for manifest inference. "
    "Run notebooks/build_datasets.ipynb first or set LOCAL_DATASET_ROOT to an existing build."
)


In [ ]:
!uv run python tools/infer_manifest_sequence.py \
  --dataset-root {LOCAL_DATASET_ROOT} \
  --manifest manifest.jsonl \
  --split val \
  --max-frames 250 \
  --init-bin /content/edge-lipsync-model/models/emma/dh_model.bin \
  --backend torch \
  --audio-wav /content/edge-lipsync-model/models/sample.wav \
  --wenet-onnx /content/edge-lipsync-model/models/wenet/wenet.onnx \
  --alpha-bin /content/edge-lipsync-model/models/emma/weight_168u.bin \
  --output-mp4 before_train.mp4 \
  --out-dir /content/data/inference/before_train \
  --device cuda


In [ ]:
from IPython.display import Video, display

display(Video("/content/data/inference/before_train/before_train.mp4", embed=True))


# Train With Console And W&B Logging


In [ ]:
!PYTHONUNBUFFERED=1 uv run python -u tools/train.py --config configs/train.colab.yaml


In [ ]:
!tail -n 20 /content/data/runs/hdtf_speaker_baseline/metrics.csv || true

import json
from pathlib import Path
metadata_path = Path("/content/data/runs/hdtf_speaker_baseline/run_metadata.json")
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    print(metadata.get("provenance", {}).get("wandb", {}))


# Optional After-Train Inference


In [ ]:
!uv run python tools/infer_manifest_sequence.py \
  --dataset-root {LOCAL_DATASET_ROOT} \
  --manifest manifest.jsonl \
  --split val \
  --max-frames 250 \
  --ckpt /content/data/runs/hdtf_speaker_baseline/best.pt \
  --backend torch \
  --audio-wav /content/edge-lipsync-model/models/sample.wav \
  --wenet-onnx /content/edge-lipsync-model/models/wenet/wenet.onnx \
  --alpha-bin /content/edge-lipsync-model/models/emma/weight_168u.bin \
  --output-mp4 after_train.mp4 \
  --out-dir /content/data/inference/after_train \
  --device cuda


In [ ]:
from IPython.display import Video, display

display(Video("/content/data/inference/after_train/after_train.mp4", embed=True))
